In [30]:

   
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report
)

feature_cols = [
    "altitude",
    "speed_trajectory",
    "signal_strength",
    "round_trip_time",
    "network_traffic_volume"
]

best_knn = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=41,
        weights="distance",
        metric="minkowski",
        p=2,
        n_jobs=-1
    ))
])

best_knn.fit(X_train[feature_cols], y_train)

y_pred = best_knn.predict(X_test[feature_cols])

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Balanced accuracy:", round(
    balanced_accuracy_score(y_test, y_pred), 4
))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(
    y_test, y_pred, zero_division=0
))


Accuracy: 0.8873
Balanced accuracy: 0.4984

Confusion matrix:
[[  10  906]
 [ 115 8027]]

Classification report:
              precision    recall  f1-score   support

           0       0.08      0.01      0.02       916
           1       0.90      0.99      0.94      8142

    accuracy                           0.89      9058
   macro avg       0.49      0.50      0.48      9058
weighted avg       0.82      0.89      0.85      9058



In [31]:

import os
import joblib

os.makedirs("droneguard_best_knn", exist_ok=True)

joblib.dump(
    {
        "model": best_knn,
        "features": feature_cols,
        "normal_label": 0,
        "anomaly_label": 1
    },
    "droneguard_best_knn/best_knn_model.joblib"
)

print("Model saved successfully.")


Model saved successfully.


In [10]:

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import joblib

logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

# Train
logistic_model.fit(X_train, y_train)

# Predict
logistic_pred = logistic_model.predict(X_test)

# Evaluate
print("=== LOGISTIC REGRESSION RESULTS ===")
print("Accuracy:", round(
    accuracy_score(y_test, logistic_pred), 4
))
print("Precision:", round(
    precision_score(y_test, logistic_pred, zero_division=0), 4
))
print("Recall:", round(
    recall_score(y_test, logistic_pred, zero_division=0), 4
))
print("F1-score:", round(
    f1_score(y_test, logistic_pred, zero_division=0), 4
))

print("\nConfusion matrix:")
print(confusion_matrix(
    y_test, logistic_pred, labels=[0, 1]
))

print("\nClassification report:")
print(classification_report(
    y_test,
    logistic_pred,
    labels=[0, 1],
    target_names=["Normal", "Anomaly"],
    zero_division=0
))

# Save trained model
joblib.dump(
    logistic_model,
    OUTPUT_DIR / "logistic_regression_model.joblib"
)

print("Logistic Regression model saved.")


=== LOGISTIC REGRESSION RESULTS ===
Accuracy: 0.6614
Precision: 0.8988
Recall: 0.7024
F1-score: 0.7886

Confusion matrix:
[[ 272  644]
 [2423 5719]]

Classification report:
              precision    recall  f1-score   support

      Normal       0.10      0.30      0.15       916
     Anomaly       0.90      0.70      0.79      8142

    accuracy                           0.66      9058
   macro avg       0.50      0.50      0.47      9058
weighted avg       0.82      0.66      0.72      9058

Logistic Regression model saved.


In [11]:

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
import joblib

tree_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", DecisionTreeClassifier(
        max_depth=3,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42
    ))
])

tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)

print("=== DECISION TREE BASELINE ===")
print("Accuracy:", round(
    accuracy_score(y_test, tree_pred), 4
))
print("Precision:", round(
    precision_score(y_test, tree_pred, zero_division=0), 4
))
print("Recall:", round(
    recall_score(y_test, tree_pred, zero_division=0), 4
))
print("F1-score:", round(
    f1_score(y_test, tree_pred, zero_division=0), 4
))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, tree_pred, labels=[0, 1]))

joblib.dump(
    tree_model,
    OUTPUT_DIR / "decision_tree_baseline.joblib"
)

print("Decision Tree baseline saved.")


=== DECISION TREE BASELINE ===
Accuracy: 0.857
Precision: 0.8984
Recall: 0.9482
F1-score: 0.9226

Confusion matrix:
[[  43  873]
 [ 422 7720]]
Decision Tree baseline saved.


In [12]:

results = []

predictions = {
    "Decision Tree": tree_pred,
    "KNN": knn_pred,
    "Logistic Regression": logistic_pred
}

for model_name, pred in predictions.items():
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(
            y_test, pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test, pred, zero_division=0
        ),
        "F1-score": f1_score(
            y_test, pred, zero_division=0
        )
    })

comparison = pd.DataFrame(results)
comparison = comparison.sort_values(
    "F1-score", ascending=False
).reset_index(drop=True)

print("=== MODEL COMPARISON ===")
display(comparison.round(4))

comparison.to_csv(
    OUTPUT_DIR / "model_comparison.csv",
    index=False
)

print("\nSaved comparison to:", OUTPUT_DIR / "model_comparison.csv")


=== MODEL COMPARISON ===


,Model,Accuracy,Precision,Recall,F1-score
0,KNN,0.8824,0.8983,0.9802,0.9375
1,Decision Tree,0.8570,0.8984,0.9482,0.9226
2,Logistic Regression,0.6614,0.8988,0.7024,0.7886



Saved comparison to: droneguard_model_comparison\model_comparison.csv


In [13]:

prediction_results = test[
    ["timestamp"] + FEATURES
].copy()

prediction_results["actual_anomaly"] = y_test.to_numpy()

for model_name, pred in predictions.items():
    column_name = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
        + "_prediction"
    )
    prediction_results[column_name] = pred

prediction_results.to_csv(
    OUTPUT_DIR / "all_model_predictions.csv",
    index=False
)

display(prediction_results.head(10))

print("Saved predictions to:",
      OUTPUT_DIR / "all_model_predictions.csv")


,timestamp,altitude,speed_trajectory,signal_strength,round_trip_time,network_traffic_volume,actual_anomaly,decision_tree_prediction,knn_prediction,logistic_regression_prediction
36231,2023-12-19 15:00:00,100,30,-60,50,100,1,1,1,1
36232,2023-12-19 16:00:00,100,30,-60,50,100,1,1,1,1
36233,2023-12-19 17:00:00,100,30,-60,50,100,1,1,1,1
36234,2023-12-19 18:00:00,100,30,-60,50,100,1,1,1,1
36235,2023-12-19 19:00:00,100,30,-60,433,576,1,1,1,0
36236,2023-12-19 20:00:00,439,30,-60,50,1235,1,1,1,0
36237,2023-12-19 21:00:00,100,87,-60,147,100,1,1,1,1
36238,2023-12-19 22:00:00,100,30,-60,346,100,0,1,1,1
36239,2023-12-19 23:00:00,100,30,-60,50,100,1,1,1,1
36240,2023-12-20 00:00:00,100,30,-60,50,100,0,1,1,1


Saved predictions to: droneguard_model_comparison\all_model_predictions.csv


In [16]:

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# 1. Build a pipeline
knn_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_jobs=-1))
])

# 2. Define hyperparameter combinations
param_grid = [
    {
        "scaler": [StandardScaler(), RobustScaler()],
        "knn__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31, 41],
        "knn__weights": ["uniform", "distance"],
        "knn__metric": ["minkowski"],
        "knn__p": [1, 2]
    },
    {
        "scaler": [StandardScaler(), RobustScaler()],
        "knn__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31, 41],
        "knn__weights": ["uniform", "distance"],
        "knn__metric": ["manhattan", "euclidean"]
    }
]

# 3. Chronological cross-validation
tscv = TimeSeriesSplit(n_splits=5)

grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    scoring="balanced_accuracy",
    cv=tscv,
    n_jobs=-1,
    verbose=1,
    refit=True
)

# 4. Train and search for the best configuration
grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)

print("\nBest mean chronological CV accuracy:")
print(round(grid_search.best_score_, 4))

# 5. Evaluate once on the untouched test set
best_knn = grid_search.best_estimator_
y_pred = best_knn.predict(X_test)

print("\nHeld-out test results")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(
    precision_score(y_test, y_pred, zero_division=0), 4
))
print("Recall:", round(
    recall_score(y_test, y_pred, zero_division=0), 4
))
print("F1-score:", round(
    f1_score(y_test, y_pred, zero_division=0), 4
))
print("Balanced accuracy:", round(
    balanced_accuracy_score(y_test, y_pred), 4
))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))


Fitting 5 folds for each of 144 candidates, totalling 720 fits
Best parameters:
{'knn__metric': 'minkowski', 'knn__n_neighbors': 41, 'knn__p': 2, 'knn__weights': 'distance', 'scaler': RobustScaler()}

Best mean chronological CV accuracy:
0.5007

Held-out test results
Accuracy: 0.8873
Precision: 0.8986
Recall: 0.9859
F1-score: 0.9402
Balanced accuracy: 0.4984

Confusion matrix:
[[  10  906]
 [ 115 8027]]

Classification report:
              precision    recall  f1-score   support

           0       0.08      0.01      0.02       916
           1       0.90      0.99      0.94      8142

    accuracy                           0.89      9058
   macro avg       0.49      0.50      0.48      9058
weighted avg       0.82      0.89      0.85      9058



In [15]:

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# Existing original model from your earlier notebook
# Assumes it is named knn_model and has already been fitted.

models = {
    "Original KNN": knn_model,
    "Tuned KNN": best_knn
}

results = []

for name, model in models.items():
    pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(
            y_test, pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test, pred, zero_division=0
        ),
        "F1-score": f1_score(
            y_test, pred, zero_division=0
        )
    })

comparison = pd.DataFrame(results)
print(comparison.round(4).to_string(index=False))


       Model  Accuracy  Precision  Recall  F1-score
Original KNN    0.8824     0.8983  0.9802    0.9375
   Tuned KNN    0.8988     0.8989  0.9999    0.9467


In [17]:
from sklearn.metrics import confusion_matrix, balanced_accuracy_score

for name, model in {
    "Original KNN": knn_model,
    "Tuned KNN": best_knn
}.items():
    predictions = model.predict(X_test)

    print(f"\n{name}")
    print(confusion_matrix(y_test, predictions))
    print(
        "Balanced accuracy:",
        round(balanced_accuracy_score(y_test, predictions), 4)
    )


Original KNN
[[  12  904]
 [ 161 7981]]
Balanced accuracy: 0.4967

Tuned KNN
[[  10  906]
 [ 115 8027]]
Balanced accuracy: 0.4984


In [18]:

import pandas as pd
import numpy as np

feature_cols = [
    "altitude",
    "speed_trajectory",
    "signal_strength",
    "round_trip_time",
    "network_traffic_volume"
]

train_df = X_train[feature_cols].copy()
train_df["target"] = np.asarray(y_train)

print("Class distribution:")
print(train_df["target"].value_counts())
print("\nClass proportions:")
print(train_df["target"].value_counts(normalize=True).round(4))

print("\nFeature statistics by class:")
print(
    train_df.groupby("target")[feature_cols]
    .agg(["mean", "median", "std"])
    .round(2)
)


Class distribution:
target
1    32539
0     3692
Name: count, dtype: int64

Class proportions:
target
1    0.8981
0    0.1019
Name: proportion, dtype: float64

Feature statistics by class:
       altitude               speed_trajectory               signal_strength  \
           mean median    std             mean median    std            mean   
target                                                                         
0        115.49  100.0  63.91            33.33   30.0  13.14          -60.30   
1        115.35  100.0  63.73            33.34   30.0  13.24          -60.46   

                    round_trip_time                network_traffic_volume  \
       median   std            mean median     std                   mean   
target                                                                      
0       -60.0  5.27          100.19   50.0  113.08                 402.18   
1       -60.0  5.17           99.83   50.0  112.36                 388.27   

                       


In [19]:

import pandas as pd
import numpy as np

feature_cols = [
    "altitude",
    "speed_trajectory",
    "signal_strength",
    "round_trip_time",
    "network_traffic_volume"
]

train_df = X_train[feature_cols].copy()
train_df["target"] = np.asarray(y_train)

print("Class distribution:")
print(train_df["target"].value_counts())
print("\nClass proportions:")
print(train_df["target"].value_counts(normalize=True).round(4))

print("\nFeature statistics by class:")
print(
    train_df.groupby("target")[feature_cols]
    .agg(["mean", "median", "std"])
    .round(2)
)


Class distribution:
target
1    32539
0     3692
Name: count, dtype: int64

Class proportions:
target
1    0.8981
0    0.1019
Name: proportion, dtype: float64

Feature statistics by class:
       altitude               speed_trajectory               signal_strength  \
           mean median    std             mean median    std            mean   
target                                                                         
0        115.49  100.0  63.91            33.33   30.0  13.14          -60.30   
1        115.35  100.0  63.73            33.34   30.0  13.24          -60.46   

                    round_trip_time                network_traffic_volume  \
       median   std            mean median     std                   mean   
target                                                                      
0       -60.0  5.27          100.19   50.0  113.08                 402.18   
1       -60.0  5.17           99.83   50.0  112.36                 388.27   

                       


In [20]:

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import RobustScaler

feature_cols = [
    "altitude",
    "speed_trajectory",
    "signal_strength",
    "round_trip_time",
    "network_traffic_volume"
]

X = X_train[feature_cols].copy()
y = pd.Series(np.asarray(y_train), index=X.index)

# Impute missing values using training data only
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=feature_cols,
    index=X.index
)

# Standardized mean difference and mutual information
results = []

for feature in feature_cols:
    normal = X_imputed.loc[y == 0, feature]
    anomaly = X_imputed.loc[y == 1, feature]

    pooled_std = np.sqrt(
        (
            (len(normal) - 1) * normal.var()
            + (len(anomaly) - 1) * anomaly.var()
        ) / (len(normal) + len(anomaly) - 2)
    )

    smd = (
        (anomaly.mean() - normal.mean()) / pooled_std
        if pooled_std > 0 else 0
    )

    results.append({
        "Feature": feature,
        "Normal mean": normal.mean(),
        "Anomaly mean": anomaly.mean(),
        "Standardized mean difference": smd
    })

mi = mutual_info_classif(
    X_imputed,
    y,
    random_state=42
)

results_df = pd.DataFrame(results)
results_df["Mutual information"] = mi
results_df["Absolute mean difference"] = (
    results_df["Standardized mean difference"].abs()
)

print(
    results_df.sort_values(
        "Mutual information",
        ascending=False
    ).round(4).to_string(index=False)
)


               Feature  Normal mean  Anomaly mean  Standardized mean difference  Mutual information  Absolute mean difference
      speed_trajectory      33.3348       33.3424                        0.0006              0.0067                    0.0006
       signal_strength     -60.2958      -60.4602                       -0.0317              0.0057                    0.0317
              altitude     115.4916      115.3527                       -0.0022              0.0025                    0.0022
       round_trip_time     100.1858       99.8300                       -0.0032              0.0018                    0.0032
network_traffic_volume     402.1804      388.2679                       -0.0255              0.0012                    0.0255


In [21]:
print(df.columns.tolist())

['timestamp', 'signal_strength', 'packet_loss_rate', 'round_trip_time', 'communication_protocol', 'frequency_band', 'encryption_type', 'drone_gps_coordinates', 'altitude', 'speed_trajectory', 'transmission_power', 'message_authentication_status', 'session_key_validity', 'signal_noise_ratio', 'sequence_number_gap', 'drone_identification', 'data_rate', 'network_traffic_volume', 'gps_signal_integrity', 'uplink_downlink_quality', 'base_station_load', 'port_scanning_attempts', 'drone_signal_handoff', 'malware_detection_signals', 'anomaly_in_behavioral_pattern', 'intrusion_detection_flags', 'temporal_patterns', 'label_normal', 'label_spoofing', 'label_mitm', 'label_ddos', 'label_gps_spoofing', 'label_malware', 'label_jamming', 'label_protocol_exploit', 'is_anomaly']


In [22]:

import pandas as pd

attack_cols = [
    "label_spoofing",
    "label_mitm",
    "label_ddos",
    "label_gps_spoofing",
    "label_malware",
    "label_jamming",
    "label_protocol_exploit"
]

# Check that label columns contain binary values
for col in ["label_normal", "is_anomaly"] + attack_cols:
    print(f"{col}: {sorted(df[col].dropna().unique().tolist())}")

# Check consistency between label_normal and is_anomaly
normal_target_mismatch = (
    df["is_anomaly"] != (1 - df["label_normal"])
)

# Check whether any attack label is active
any_attack = df[attack_cols].eq(1).any(axis=1)

# Rows marked normal but also assigned an attack label
normal_with_attack = (
    df["label_normal"].eq(1) & any_attack
)

# Rows marked anomalous but with no attack label active
anomaly_without_attack = (
    df["label_normal"].eq(0) & ~any_attack
)

print("\nTotal rows:", len(df))
print("is_anomaly vs 1 - label_normal mismatches:",
      int(normal_target_mismatch.sum()))
print("Normal rows with an attack label:",
      int(normal_with_attack.sum()))
print("Anomalous rows without an attack label:",
      int(anomaly_without_attack.sum()))

print("\nAttack-label combinations:")
print(
    df[attack_cols].value_counts(dropna=False).head(15)
)


label_normal: [0, 1]
is_anomaly: [0, 1]
label_spoofing: [0, 1]
label_mitm: [0, 1]
label_ddos: [0, 1]
label_gps_spoofing: [0, 1]
label_malware: [0, 1]
label_jamming: [0, 1]
label_protocol_exploit: [0, 1]

Total rows: 45289
is_anomaly vs 1 - label_normal mismatches: 0
Normal rows with an attack label: 1375
Anomalous rows without an attack label: 28225

Attack-label combinations:
label_spoofing  label_mitm  label_ddos  label_gps_spoofing  label_malware  label_jamming  label_protocol_exploit
0               0           0           0                   0              0              0                         31458
                            1           0                   0              0              0                          2770
                            0           0                   0              1              0                          2343
                                                            1              0              0                          2040
1               0  

In [24]:


import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif

baseline_features = [
    "altitude",
    "speed_trajectory",
    "signal_strength",
    "round_trip_time",
    "network_traffic_volume"
]

# Start with raw telemetry and communication measurements.
candidate_features = [
    "packet_loss_rate",
    "transmission_power",
    "signal_noise_ratio",
    "sequence_number_gap",
    "data_rate",
    "base_station_load",
    "uplink_downlink_quality"
]

features = baseline_features + candidate_features

# Confirm that the required columns exist.
missing = [c for c in features if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# This assumes X_train retains original df row indices.
# It is essential that X_train and y_train represent the same
# chronological training split used in previous experiments.
X_screen = df.loc[X_train.index, features].copy()
y_screen = pd.Series(
    y_train.to_numpy() if hasattr(y_train, "to_numpy")
    else y_train,
    index=X_screen.index
)

# Keep numeric features for this initial experiment.
X_screen = X_screen.select_dtypes(include="number")

# Remove columns with no variation.
X_screen = X_screen.loc[:, X_screen.nunique(dropna=True) > 1]

# Impute using training data only.
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(
    imputer.fit_transform(X_screen),
    columns=X_screen.columns,
    index=X_screen.index
)

mi = mutual_info_classif(
    X_imputed,
    y_screen,
    random_state=42
)

ranking = (
    pd.DataFrame({
        "Feature": X_screen.columns,
        "Mutual information": mi
    })
    .sort_values("Mutual information", ascending=False)
    .reset_index(drop=True)
)

print("Training rows:", len(X_screen))
print("\nFeature ranking:")
print(ranking.round(5).to_string(index=False))



Training rows: 36231

Feature ranking:
                Feature  Mutual information
       speed_trajectory             0.00649
     transmission_power             0.00621
uplink_downlink_quality             0.00581
              data_rate             0.00536
     signal_noise_ratio             0.00484
        signal_strength             0.00471
      base_station_load             0.00439
               altitude             0.00309
 network_traffic_volume             0.00213
       packet_loss_rate             0.00145
        round_trip_time             0.00000
    sequence_number_gap             0.00000


In [27]:

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

baseline_features = [
    "altitude",
    "speed_trajectory",
    "signal_strength",
    "round_trip_time",
    "network_traffic_volume"
]

expanded_features = baseline_features + [
    "transmission_power",
    "uplink_downlink_quality",
    "data_rate",
    "signal_noise_ratio",
    "base_station_load",
    "packet_loss_rate"
]

# Ensure the rows align with the existing chronological training split.
X_base = df.loc[X_train.index, baseline_features].copy()
X_expanded = df.loc[X_train.index, expanded_features].copy()

y = pd.Series(
    y_train.to_numpy() if hasattr(y_train, "to_numpy") else y_train,
    index=X_train.index
)

# Verify alignment.
assert X_base.index.equals(y.index)
assert X_expanded.index.equals(y.index)

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
    ("knn", KNeighborsClassifier(n_jobs=-1))
])

param_grid = {
    "knn__n_neighbors": [11, 21, 31, 41, 51],
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2]
}

cv = TimeSeriesSplit(n_splits=5)
results = []

for name, X in [
    ("Baseline: 5 features", X_base),
    ("Expanded: 11 features", X_expanded)
]:
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="balanced_accuracy",
        cv=cv,
        n_jobs=-1,
        refit=True
    )

    search.fit(X, y)

    results.append({
        "Feature set": name,
        "Best CV balanced accuracy": search.best_score_,
        "Best parameters": search.best_params_
    })

print(pd.DataFrame(results).to_string(index=False))


          Feature set  Best CV balanced accuracy                                                   Best parameters
 Baseline: 5 features                   0.500687 {'knn__n_neighbors': 41, 'knn__p': 2, 'knn__weights': 'distance'}
Expanded: 11 features                   0.500422 {'knn__n_neighbors': 51, 'knn__p': 2, 'knn__weights': 'distance'}


In [28]:

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

tree_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("tree", DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ))
])

tree_params = {
    "tree__max_depth": [3, 5, 8, 12, None],
    "tree__min_samples_leaf": [10, 25, 50, 100],
    "tree__criterion": ["gini", "entropy"]
}

tree_search = GridSearchCV(
    estimator=tree_pipeline,
    param_grid=tree_params,
    scoring="balanced_accuracy",
    cv=TimeSeriesSplit(n_splits=5),
    n_jobs=-1,
    refit=True,
    verbose=1
)

tree_search.fit(X_expanded, y)

print("Best parameters:")
print(tree_search.best_params_)

print(
    "\nBest CV balanced accuracy:",
    round(tree_search.best_score_, 4)
)


Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best parameters:
{'tree__criterion': 'entropy', 'tree__max_depth': None, 'tree__min_samples_leaf': 10}

Best CV balanced accuracy: 0.511
